In [49]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder, KBinsDiscretizer
from imblearn.over_sampling import SMOTE

from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.neighbors import NearestNeighbors
from scipy.cluster.hierarchy import dendrogram, linkage

from mlxtend.frequent_patterns import apriori, association_rules

In [50]:
df = pd.read_csv("Sleep_health_and_lifestyle_dataset.csv")

print(f" {df} Columns ")
print(list(df.columns))


      Person ID  Gender  Age            Occupation  Sleep Duration  \
0            1    Male   27     Software Engineer             6.1   
1            2    Male   28                Doctor             6.2   
2            3    Male   28                Doctor             6.2   
3            4    Male   28  Sales Representative             5.9   
4            5    Male   28  Sales Representative             5.9   
..         ...     ...  ...                   ...             ...   
369        370  Female   59                 Nurse             8.1   
370        371  Female   59                 Nurse             8.0   
371        372  Female   59                 Nurse             8.1   
372        373  Female   59                 Nurse             8.1   
373        374  Female   59                 Nurse             8.1   

     Quality of Sleep  Physical Activity Level  Stress Level BMI Category  \
0                   6                       42             6   Overweight   
1               

In [51]:
print(f"{len(df)} ")

rows, columns = df.shape
print(f"Το dataset έχει {rows} γραμμές και {columns} στήλες.")

374 
Το dataset έχει 374 γραμμές και 13 στήλες.


In [52]:
pd.set_option('display.max_columns', None)
display(df.head())

,Person ID,Gender,Age,Occupation,Sleep Duration,Quality of Sleep,Physical Activity Level,Stress Level,BMI Category,Blood Pressure,Heart Rate,Daily Steps,Sleep Disorder
0,1,Male,27,Software Engineer,6.1,6,42,6,Overweight,126/83,77,4200,NaN
1,2,Male,28,Doctor,6.2,6,60,8,Normal,125/80,75,10000,NaN
2,3,Male,28,Doctor,6.2,6,60,8,Normal,125/80,75,10000,NaN
3,4,Male,28,Sales Representative,5.9,4,30,8,Obese,140/90,85,3000,Sleep Apnea
4,5,Male,28,Sales Representative,5.9,4,30,8,Obese,140/90,85,3000,Sleep Apnea


## ΚΑΘΑΡΙΣΜΟΣ ΔΕΔΟΜΕΝΩΝ

Αρχικά, θα διαχωρίσω την πίεση του αίματος σε συστολική και διαστολική χωρίζοντας τη στήλη Blood Pressure σε 2 στήλες:

In [53]:
df[['Systolic', 'Diastolic']] = df['Blood Pressure'].str.split('/', expand=True)

df['Systolic'] = pd.to_numeric(df['Systolic'])
df['Diastolic'] = pd.to_numeric(df['Diastolic'])

df = df.drop('Blood Pressure', axis=1)
display(df.head())

,Person ID,Gender,Age,Occupation,Sleep Duration,Quality of Sleep,Physical Activity Level,Stress Level,BMI Category,Heart Rate,Daily Steps,Sleep Disorder,Systolic,Diastolic
0,1,Male,27,Software Engineer,6.1,6,42,6,Overweight,77,4200,NaN,126,83
1,2,Male,28,Doctor,6.2,6,60,8,Normal,75,10000,NaN,125,80
2,3,Male,28,Doctor,6.2,6,60,8,Normal,75,10000,NaN,125,80
3,4,Male,28,Sales Representative,5.9,4,30,8,Obese,85,3000,Sleep Apnea,140,90
4,5,Male,28,Sales Representative,5.9,4,30,8,Obese,85,3000,Sleep Apnea,140,90


In [54]:
print(df['BMI Category'].unique())
df['BMI Category'] = df['BMI Category'].replace('Normal Weight', 'Normal')
display(df.head())

<StringArray>
['Overweight', 'Normal', 'Obese', 'Normal Weight']
Length: 4, dtype: str


,Person ID,Gender,Age,Occupation,Sleep Duration,Quality of Sleep,Physical Activity Level,Stress Level,BMI Category,Heart Rate,Daily Steps,Sleep Disorder,Systolic,Diastolic
0,1,Male,27,Software Engineer,6.1,6,42,6,Overweight,77,4200,NaN,126,83
1,2,Male,28,Doctor,6.2,6,60,8,Normal,75,10000,NaN,125,80
2,3,Male,28,Doctor,6.2,6,60,8,Normal,75,10000,NaN,125,80
3,4,Male,28,Sales Representative,5.9,4,30,8,Obese,85,3000,Sleep Apnea,140,90
4,5,Male,28,Sales Representative,5.9,4,30,8,Obese,85,3000,Sleep Apnea,140,90


Παρατηρώντας τα Δεδομένα οι στήλες Gender Occupation BMICategory SleepDisorder είναι ποιοτικές μεταβλητές
Ενώ οι Age SleepDuration QualityofSleep PhysicalActivityLevel StressLevel HeartRate DailySteps Systolic Diastolic είναι ποσοτικές.

Οι αλγόριθμοι καταλαβαίνουν μόνο αριθμούς και αποστάσεις. Άρα πρέπει να μετατρέψουμε τις ποιοτικές μεταβλημτές σε ποσοτικές.

In [55]:
categorical_cols = ['Gender', 'Occupation', 'BMI Category']
df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

display(df.head())

,Person ID,Age,Sleep Duration,Quality of Sleep,Physical Activity Level,Stress Level,Heart Rate,Daily Steps,Sleep Disorder,Systolic,Diastolic,Gender_Male,Occupation_Doctor,Occupation_Engineer,Occupation_Lawyer,Occupation_Manager,Occupation_Nurse,Occupation_Sales Representative,Occupation_Salesperson,Occupation_Scientist,Occupation_Software Engineer,Occupation_Teacher,BMI Category_Obese,BMI Category_Overweight
0,1,27,6.1,6,42,6,77,4200,NaN,126,83,True,False,False,False,False,False,False,False,False,True,False,False,True
1,2,28,6.2,6,60,8,75,10000,NaN,125,80,True,True,False,False,False,False,False,False,False,False,False,False,False
2,3,28,6.2,6,60,8,75,10000,NaN,125,80,True,True,False,False,False,False,False,False,False,False,False,False,False
3,4,28,5.9,4,30,8,85,3000,Sleep Apnea,140,90,True,False,False,False,False,False,True,False,False,False,False,True,False
4,5,28,5.9,4,30,8,85,3000,Sleep Apnea,140,90,True,False,False,False,False,False,True,False,False,False,False,True,False


In [56]:
print("Κενές τιμές ανά στήλη:")
print(df.isnull().sum())

duplicates_count = df.duplicated().sum()
print(f"Συνολικές διπλότυπες γραμμές στο dataset: {duplicates_count}")

Κενές τιμές ανά στήλη:
Person ID                            0
Age                                  0
Sleep Duration                       0
Quality of Sleep                     0
Physical Activity Level              0
Stress Level                         0
Heart Rate                           0
Daily Steps                          0
Sleep Disorder                     219
Systolic                             0
Diastolic                            0
Gender_Male                          0
Occupation_Doctor                    0
Occupation_Engineer                  0
Occupation_Lawyer                    0
Occupation_Manager                   0
Occupation_Nurse                     0
Occupation_Sales Representative      0
Occupation_Salesperson               0
Occupation_Scientist                 0
Occupation_Software Engineer         0
Occupation_Teacher                   0
BMI Category_Obese                   0
BMI Category_Overweight              0
dtype: int64
Συνολικές διπλότυπες γραμμές

In [57]:
df['Sleep Disorder'] = df['Sleep Disorder'].fillna('None')

print("Κατανομή των κατηγοριών διαταραχής ύπνου:")
print(df['Sleep Disorder'].value_counts())

Κατανομή των κατηγοριών διαταραχής ύπνου:
Sleep Disorder
None           219
Sleep Apnea     78
Insomnia        77
Name: count, dtype: int64
